In [ ]:
# code mostly by @light-weaver, some edits by @maiemile

import pandas as pd
import plotly.express as ex
import plotly.graph_objects as go

import os

In [ ]:
files = os.listdir("archived_final_pops_v3")
files

files = [file.split(".")[0].split("-") for file in files]
files = [file for file in files if file[1] =="3obj"]
files = pd.DataFrame(files, columns=["problem", "num_obj", "algorithm", "xover", "mut"])
files

In [ ]:
files["problem"].unique()

In [ ]:
inds = pd.read_csv("igd_values_log.txt", sep=" ", header=None)
inds.columns = ["filename", "igd", "igdplus"]
inds = pd.concat([
    inds, 
    pd.DataFrame([file.split("-") for file in inds["filename"]], columns= ["problem", "num_obj", "algorithm", "xover", "mut"])
    ], axis=1)

inds = inds[inds["num_obj"]=="3obj"]

inds["problem"].unique()

In [ ]:
inds = inds[inds['igdplus'].notna()]

In [ ]:
inds['igd'] = inds['igd'].apply(pd.to_numeric)

In [ ]:
inds.info()

In [ ]:
inds['algorithm'] = inds['algorithm'].replace(["rvea", "nsga3", "ibea"], ["RVEA", "NSGA-III", "IBEA"])

In [ ]:
inds['xover'] = inds['xover'].replace(["Local", "Balpha", "Single"], ["LX", "BLX-α", "SAX"])

In [ ]:
inds

In [ ]:
for problem in inds["problem"].unique():
    subset = inds[inds["problem"]==problem]
    subset = subset.sort_values(by=["algorithm", "xover"])
    boxplot= ex.box(subset, y="igd", x="algorithm", color="xover", 
                    labels={
                     "igd": "IGD",
                     "algorithm": "Algorithm",
                     "xover": "Crossover operator"
                 },
                    title=f"Boxplots of EA configuration performance on {problem.upper()}")
    boxplot.update_layout(
        font=dict(
            size=14, 
            color="Black"
        )
    )
    boxplot.show()
    boxplot.write_html(f"boxplot_{problem}.html")
    xovers = ["SBX", "Single", "Local", "Balpha"]
    algos = ["nsga3", "rvea", "ibea"]
    subset = subset[[xover in xovers for xover in subset["xover"]]]
    subset = subset.sort_values(by="igd").reset_index()

    fig = ex.scatter_3d()
    ref = pd.read_csv(f"approx_pfs_v3/{problem}-3obj.txt", sep=" ", header=None)
    ref.columns=["f1", "f2", "f3"]
    fig.add_scatter3d(
        x=ref["f1"],
        y=ref["f2"],
        z=ref["f3"],
        mode="markers",
        name=f"Reference",
        marker_size=2,
        opacity=0.5
    )
    for i, file in subset.iterrows():
        try:
            data = pd.read_csv(f"archived_pops_v3/{file["filename"]}.txt", sep=" ", header=None)
        except:
            continue
        data.columns=["f1", "f2", "f3"]
        fig.add_scatter3d(
            x=data["f1"],
            y=data["f2"],
            z=data["f3"],
            mode="markers",
            name=f"igd={file['igd']} config={file['filename']}"
        )

    fig.update_layout(title=f"PF Approximations for various EA configs for {problem}")
    fig.update_layout(scene_aspectmode='cube')
    fig.write_html(f"PF_full_archive_{problem}.html")